# Install Libraries

In [ ]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install wordcloud
!pip install scikit-learn
!pip install langchain langchain-community

: 

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud

from sentence_transformers import SentenceTransformer

import faiss

from sklearn.decomposition import PCA

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS as LangchainFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/janakiram1700/flipkart-sample/e827e52d-d4e3-4a34-8981-7ce843253733_Dataset-SA (1).csv")
df.shape
df.columns
df.head()

# Data Cleaning

In [ ]:
df.dropna(subset=["Review","Summary"], inplace=True)
df.drop_duplicates(subset=["Review","Summary"], inplace=True)
df["text"] = df["Summary"] + " " + df["Review"]
df.head()

# Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x="Rate", data=df)

plt.title("Distribution of Product Ratings")
plt.show()

In [ ]:
df["text_length"] = df["text"].apply(len)

plt.figure(figsize=(8,5))

sns.histplot(df["text_length"], bins=50)

plt.title("Review Length Distribution")
plt.show()

In [ ]:
all_text = " ".join(df["text"].sample(5000).astype(str))

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(all_text)

plt.figure(figsize=(10,6))

plt.imshow(wordcloud)

plt.axis("off")

plt.title("Word Cloud of Reviews")

plt.show()

# Generate Sentence Embeddings

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
df["Rate"] = pd.to_numeric(df["Rate"], errors="coerce")
sampled_df["Rate"] = pd.to_numeric(sampled_df["Rate"], errors="coerce")

embeddings = model.encode(
    sampled_df["text"].tolist(),
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

# PCA Visualization

In [ ]:
pca = PCA(n_components=2)

reduced_embeddings = pca.fit_transform(embeddings)

plt.figure(figsize=(6,4))

scatter = plt.scatter(
    reduced_embeddings[:,0],
    reduced_embeddings[:,1],
    c=sampled_df["Rate"],
    cmap="viridis",
    alpha=0.6
)

plt.colorbar(scatter, label="Rating")

plt.title("PCA Visualization of Review Embeddings")

plt.show()

# Normalize Embeddings

In [ ]:
embeddings_normalized = embeddings.copy()

faiss.normalize_L2(embeddings_normalized)

# Build FAISS Vector Index

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings_normalized)

print("Total vectors in index:", index.ntotal)

# Semantic Search Function

In [ ]:
def search_reviews(query, top_k=5):

    query_vector = model.encode([query])

    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, top_k)

    results = []

    for i, idx in enumerate(indices[0]):

        results.append({

            "similarity_score": scores[0][i],
            "product": sampled_df.iloc[idx]["product_name"],
            "review": sampled_df.iloc[idx]["Review"],
            "summary": sampled_df.iloc[idx]["Summary"],
            "rating": sampled_df.iloc[idx]["Rate"]

        })

    return results

# Test Semantic Search

In [ ]:
queries = [
"good cooling",
"bad product",
"worth the price",
"great quality",
"poor performance"
]

for query in queries:

    print("\nQuery:", query)

    results = search_reviews(query, top_k=3)

    for r in results:

        print("Similarity:", r["similarity_score"])
        print("Product:", r["product"])
        print("Rating:", r["rating"])
        print("Summary:", r["summary"])
        print("Review:", r["review"])
        print()

# Build RAG Pipeline (LangChain)

In [ ]:
documents = []

for _, row in sampled_df.iterrows():

    doc = Document(
        page_content=row["text"],
        metadata={
            "product": row["product_name"],
            "rating": row["Rate"]
        }
    )

    documents.append(doc)

In [ ]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vector_store = LangchainFAISS.from_documents(
    documents,
    embeddings_model
)

# Query RAG System

In [ ]:
query = "good cooling air cooler"

docs = vector_store.similarity_search(query, k=3)

for doc in docs:

    print("Product:", doc.metadata["product"])
    print("Rating:", doc.metadata["rating"])
    print("Text:", doc.page_content[:300])
    print()

# Evaluation Metric (Precision@K)

In [ ]:
def precision_at_k(query, k=5):

    results = search_reviews(query, k)

    relevant = [r for r in results if r["rating"] >= 4]

    return len(relevant) / k

In [ ]:
precision_at_k("good cooling",5)

In [ ]:
search_reviews("good cooling")

# search_reviews("bad quality")

# search_reviews("value for money")

# search_reviews("powerful air flow")

Business Insights

Semantic search significantly improves product discovery in e-commerce platforms by understanding user intent rather than relying on exact keyword matching. Using Sentence Transformers, customer reviews were converted into dense embeddings that capture the semantic meaning of text. The FAISS vector index enabled efficient similarity search across thousands of reviews.

Exploratory data analysis revealed that most reviews have ratings between 4 and 5, indicating generally positive customer sentiment. Word cloud analysis highlighted frequently used words such as “good”, “quality”, and “product”, suggesting customers emphasize product reliability and value.

The implemented RAG pipeline retrieves relevant reviews and generates concise summaries to help users quickly understand product performance. This system can improve search accuracy, enhance customer experience, and increase product conversion rates in e-commerce platforms.